# Qwen3 1.7B — Stateful KV-Cache Core ML Export: Other Approaches

---

## 4. Approach B: Wrapper + `torch.export`

> **Rationale**: `torch.export` is the modern replacement for `torch.jit.trace`.
> It already works for the stateless Qwen3 model in the existing notebook.
> The question is whether it correctly captures `register_buffer` mutations.

### 4.1 Why torch.export may work

`torch.export` in PyTorch 2.6 captures buffer mutations as graph mutations.
When a buffer is modified in-place (slice assignment), the exported program's
graph records these mutations. `coremltools 8.0` maps graph mutations on
registered buffers to Core ML state read/write operations when `states=` is provided.

### 4.2 Stateful model for export (modified forward signature)

---

## 5. Approach C: KV-Cache-as-I/O (Explicit Inputs/Outputs)

> **Rationale**: Entirely skip MLState. Make KV caches explicit model I/O.
> Simpler, guaranteed to work, but ~13x slower than stateful approach (per Apple benchmarks)
> because cache data must be copied to/from the model each step.

### 5.1 Qwen3 with explicit KV cache I/O

In [ ]:
class Qwen3WithExplicitKVCache(torch.nn.Module):
    """
    Qwen3 wrapper where the KV cache is an explicit input/output pair.
    
    Inputs:  input_ids, causal_mask, key_cache_in, value_cache_in
    Outputs: logits, key_cache_out, value_cache_out
    
    No register_buffer, no MLState — pure functional.
    """
    
    def __init__(
        self,
        model_path: str,
        max_context_size: int = 2048,
        batch_size: int = 1,
    ) -> None:
        super().__init__()
        
        import transformers.models.qwen3.modeling_qwen3 as qwen3_module
        original_attn_class = qwen3_module.Qwen3Attention
        qwen3_module.Qwen3Attention = SliceUpdateQwen3Attention
        
        self.model = Qwen3ForCausalLM.from_pretrained(
            model_path,
            torch_dtype=DTYPE,
            attn_implementation="eager",
            low_cpu_mem_usage=True,
        )
        self.model.eval()
        
        qwen3_module.Qwen3Attention = original_attn_class
        
        config = self.model.config
        self.kv_cache_shape = (
            config.num_hidden_layers,
            batch_size,
            config.num_key_value_heads,
            max_context_size,
            config.hidden_size // config.num_attention_heads,
        )
    
    @torch.no_grad()
    def forward(
        self,
        input_ids: torch.LongTensor,
        causal_mask: torch.Tensor,
        key_cache_in: torch.Tensor,
        value_cache_in: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns (logits, key_cache_out, value_cache_out).
        The caller manages the cache externally.
        """
        # Create cache wrapper around the input tensors
        cache = SliceUpdateKeyValueCache.__new__(SliceUpdateKeyValueCache)
        Cache.__init__(cache)
        cache.past_seen_tokens = causal_mask.shape[-1] - input_ids.shape[-1]
        cache.k_cache = key_cache_in
        cache.v_cache = value_cache_in
        
        logits = self.model(
            input_ids,
            attention_mask=causal_mask,
            past_key_values=cache,
            use_cache=True,
        ).logits
        
        return logits, cache.k_cache, cache.v_cache

### 5.2 Trace the explicit-I/O model

In [ ]:
explicit_model = Qwen3WithExplicitKVCache(
    MODEL_ID,
    max_context_size=MAX_CONTEXT_SIZE,
)
explicit_model.eval()

# Trace inputs
trace_ids = torch.zeros((1, 2), dtype=torch.int32)
trace_mask = torch.zeros((1, 1, 2, 5), dtype=DTYPE)
trace_k = torch.zeros(explicit_model.kv_cache_shape, dtype=DTYPE)
trace_v = torch.zeros(explicit_model.kv_cache_shape, dtype=DTYPE)

print("Tracing explicit KV cache model...")
traced_explicit = torch.jit.trace(
    explicit_model,
    [trace_ids, trace_mask, trace_k, trace_v],
)
print("✅ Traced successfully")

### 5.3 Convert to Core ML (no states)

In [ ]:
kv_shape = explicit_model.kv_cache_shape
del explicit_model

query_length_c = ct.RangeDim(lower_bound=1, upper_bound=MAX_CONTEXT_SIZE, default=1)
end_step_dim_c = ct.RangeDim(lower_bound=1, upper_bound=MAX_CONTEXT_SIZE, default=1)

inputs_c = [
    ct.TensorType(shape=(1, query_length_c), dtype=np.int32, name="inputIds"),
    ct.TensorType(
        shape=(1, 1, query_length_c, end_step_dim_c),
        dtype=np.float16,
        name="causalMask",
    ),
    ct.TensorType(shape=kv_shape, dtype=np.float16, name="keyCacheIn"),
    ct.TensorType(shape=kv_shape, dtype=np.float16, name="valueCacheIn"),
]

outputs_c = [
    ct.TensorType(dtype=np.float16, name="logits"),
    ct.TensorType(dtype=np.float16, name="keyCacheOut"),
    ct.TensorType(dtype=np.float16, name="valueCacheOut"),
]

print("Converting to Core ML (no states, explicit I/O)...")
mlmodel_c = ct.convert(
    traced_explicit,
    inputs=inputs_c,
    outputs=outputs_c,
    minimum_deployment_target=ct.target.iOS18,
    skip_model_load=True,
)

mlmodel_c.save("models/Qwen3_1_7B_explicit_kv_fp16.mlpackage")
print("✅ Core ML conversion complete (Approach C)")

del traced_explicit

### 5.4 Inference loop for explicit I/O approach (Python reference)

In [ ]:
"""
Reference inference loop for Approach C.
In the iOS app, this would be in Swift with MLMultiArray.

NOTE: This approach copies ~112 MB of KV cache data every forward pass.
Apple's benchmarks show stateful approach is ~13x faster.
"""

def generate_explicit_kv(mlmodel, tokenizer, prompt, max_new_tokens=50):
    tokens = tokenizer(prompt, return_tensors="np")["input_ids"].astype(np.int32)
    seq_len = tokens.shape[1]
    
    # Initialize empty KV cache
    k_cache = np.zeros(kv_shape, dtype=np.float16)
    v_cache = np.zeros(kv_shape, dtype=np.float16)
    
    generated = list(tokens[0])
    
    # Prefill
    mask = build_causal_mask(seq_len, seq_len)
    result = mlmodel.predict({
        "inputIds": tokens,
        "causalMask": mask,
        "keyCacheIn": k_cache,
        "valueCacheIn": v_cache,
    })
    k_cache = result["keyCacheOut"]
    v_cache = result["valueCacheOut"]
    next_token = int(np.argmax(result["logits"][0, -1, :]))
    generated.append(next_token)
    
    # Decode loop
    for step in range(max_new_tokens - 1):
        past_len = seq_len + step + 1
        token_input = np.array([[next_token]], dtype=np.int32)
        mask = build_causal_mask(1, past_len)
        
        result = mlmodel.predict({
            "inputIds": token_input,
            "causalMask": mask,
            "keyCacheIn": k_cache,
            "valueCacheIn": v_cache,
        })
        k_cache = result["keyCacheOut"]
        v_cache = result["valueCacheOut"]
        next_token = int(np.argmax(result["logits"][0, -1, :]))
        generated.append(next_token)
        
        if next_token == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated)

In [ ]:
class StatefulQwen3ForExport(torch.nn.Module):
    """
    Same as StatefulQwen3ForCausalLM but with a forward() signature
    compatible with torch.export (no **kwargs, explicit shapes).
    """
    
    def __init__(
        self,
        model_path: str,
        max_context_size: int = 2048,
        batch_size: int = 1,
    ) -> None:
        super().__init__()
        
        # Monkey-patch attention
        import transformers.models.qwen3.modeling_qwen3 as qwen3_module
        original_attn_class = qwen3_module.Qwen3Attention
        qwen3_module.Qwen3Attention = SliceUpdateQwen3Attention
        
        self.model = Qwen3ForCausalLM.from_pretrained(
            model_path,
            torch_dtype=DTYPE,
            attn_implementation="eager",
            low_cpu_mem_usage=True,
        )
        self.model.eval()
        
        qwen3_module.Qwen3Attention = original_attn_class
        
        config: Qwen3Config = self.model.config
        self.kv_cache_shape = (
            config.num_hidden_layers,
            batch_size,
            config.num_key_value_heads,
            max_context_size,
            config.hidden_size // config.num_attention_heads,
        )
        
        self.kv_cache = SliceUpdateKeyValueCache(
            shape=self.kv_cache_shape,
            dtype=DTYPE,
        )
        
        self.register_buffer("keyCache", self.kv_cache.k_cache)
        self.register_buffer("valueCache", self.kv_cache.v_cache)
    
    @torch.no_grad()
    def forward(
        self,
        input_ids: torch.LongTensor,
        causal_mask: torch.Tensor,
    ) -> torch.Tensor:
        self.kv_cache.past_seen_tokens = (
            causal_mask.shape[-1] - input_ids.shape[-1]
        )
        return self.model(
            input_ids,
            attention_mask=causal_mask,
            past_key_values=self.kv_cache,
            use_cache=True,
        ).logits

### 4.3 Export with torch.export

In [ ]:
stateful_export_model = StatefulQwen3ForExport(
    MODEL_ID,
    max_context_size=MAX_CONTEXT_SIZE,
)
stateful_export_model.eval()

# Example inputs for export
example_input_ids = torch.zeros((1, 32), dtype=torch.long)
example_mask = build_causal_mask_torch(32, 32, dtype=DTYPE)

from torch.export import export, Dim

seq_dim = Dim("seq_len", min=1, max=MAX_CONTEXT_SIZE)
kv_dim = Dim("kv_len", min=1, max=MAX_CONTEXT_SIZE)

print("Running torch.export.export()...")
try:
    exported_program = export(
        stateful_export_model,
        (example_input_ids, example_mask),
        dynamic_shapes={
            "input_ids": {1: seq_dim},
            "causal_mask": {2: seq_dim, 3: kv_dim},
        },
    )
    exported_program = exported_program.run_decompositions({})
    print("✅ torch.export succeeded")
    
    # Check for buffer mutations in the graph
    graph_str = str(exported_program.graph_module.graph)
    has_mutations = "mutate" in graph_str.lower() or "buffer" in graph_str.lower()
    print(f"Graph has buffer-related ops: {has_mutations}")
    
except Exception as e:
    print(f"❌ torch.export failed: {e}")
    print("This may be due to in-place buffer mutations not being supported.")
    print("Fallback: Use Approach A (torch.jit.trace) instead.")
    exported_program = None

### 4.4 Convert exported program to Core ML

In [ ]:
if exported_program is not None:
    kv_cache_shape_b = stateful_export_model.kv_cache_shape
    del stateful_export_model
    
    states_b = [
        ct.StateType(
            wrapped_type=ct.TensorType(
                shape=kv_cache_shape_b,
                dtype=np.float16,
            ),
            name="keyCache",
        ),
        ct.StateType(
            wrapped_type=ct.TensorType(
                shape=kv_cache_shape_b,
                dtype=np.float16,
            ),
            name="valueCache",
        ),
    ]
    
    print("Converting exported program to Core ML...")
    try:
        mlmodel_b = ct.convert(
            exported_program,
            convert_to="mlprogram",
            states=states_b,
            minimum_deployment_target=ct.target.iOS18,
            skip_model_load=True,
        )
        mlmodel_b.save("models/Qwen3_1_7B_stateful_export_fp16.mlpackage")
        print("✅ Core ML conversion complete (Approach B)")
        
    except Exception as e:
        print(f"❌ Core ML conversion failed: {e}")
        print("The export graph may not have buffer mutations in a form coremltools expects.")
        mlmodel_b = None
else:
    mlmodel_b = None
    print("Skipping Core ML conversion (export failed)")

---

## 6. Approach D: HuggingFace `StaticCache` + `torch.export`

> **Rationale**: Use HuggingFace's built-in `StaticCache` which pre-allocates
> fixed-size KV cache tensors. This is designed for `torch.compile` and `torch.export`.
> Combined with the `register_buffer` pattern, it might integrate with Core ML states.

### 6.1 Understanding StaticCache

In [ ]:
from transformers.cache_utils import StaticCache

"""
StaticCache pre-allocates tensors of shape:
  (batch_size, num_heads, max_cache_len, head_dim)
per layer, and mutates them in-place via index assignment.

This IS compatible with torch.export and torch.compile.
The question is: can we combine it with register_buffer + ct.StateType?
"""

# Create a StaticCache for our model config
static_cache = StaticCache(
    config=base_model.config,
    max_cache_len=MAX_CONTEXT_SIZE,
)

print(f"Number of cache layers: {len(static_cache._cache_layers)}")
layer0 = static_cache._cache_layers[0]
print(f"Layer 0 key shape: {layer0.key_cache.shape if hasattr(layer0, 'key_cache') else 'not initialized (lazy)'}")
print(f"Type: {type(layer0)}")

### 6.2 StaticCache wrapper with register_buffer

In [ ]:
class StaticCacheQwen3Wrapper(torch.nn.Module):
    """
    Uses HuggingFace's native StaticCache mechanism.
    
    The model is loaded with use_cache=True and we pass a StaticCache
    that pre-allocates the KV buffers. We then register those buffers
    so they become Core ML states.
    
    This approach avoids custom attention classes entirely — it uses
    the standard HuggingFace code path.
    """
    
    def __init__(
        self,
        model_path: str,
        max_context_size: int = 2048,
    ) -> None:
        super().__init__()
        
        self.model = Qwen3ForCausalLM.from_pretrained(
            model_path,
            torch_dtype=DTYPE,
            attn_implementation="eager",
            low_cpu_mem_usage=True,
        )
        self.model.eval()
        self.model.config.use_cache = True
        
        # Create static cache
        self.static_cache = StaticCache(
            config=self.model.config,
            max_cache_len=max_context_size,
            batch_size=1,
        )
        
        # Force eager initialization of all layers
        # StaticCache uses lazy init — we need tensors now for register_buffer
        dummy_k = torch.zeros(1, NUM_KV_HEADS, 1, HEAD_DIM, dtype=DTYPE)
        dummy_v = torch.zeros(1, NUM_KV_HEADS, 1, HEAD_DIM, dtype=DTYPE)
        for layer_idx in range(NUM_LAYERS):
            self.static_cache.update(dummy_k, dummy_v, layer_idx)
        self.static_cache.reset()
        
        # Register each layer's cache as a buffer
        for layer_idx in range(NUM_LAYERS):
            layer = self.static_cache._cache_layers[layer_idx]
            self.register_buffer(
                f"key_cache_{layer_idx}",
                layer.key_cache,
            )
            self.register_buffer(
                f"value_cache_{layer_idx}",
                layer.value_cache,
            )
    
    @torch.no_grad()
    def forward(self, input_ids: torch.LongTensor) -> torch.Tensor:
        return self.model(
            input_ids,
            past_key_values=self.static_cache,
            use_cache=True,
        ).logits

### 6.3 Attempt export with StaticCache

In [ ]:
print("Loading StaticCacheQwen3Wrapper...")
static_wrapper = StaticCacheQwen3Wrapper(
    MODEL_ID,
    max_context_size=MAX_CONTEXT_SIZE,
)
static_wrapper.eval()

print(f"Registered buffers: {len(list(static_wrapper.named_buffers()))}")

# Try torch.export
example = (torch.zeros((1, 32), dtype=torch.long),)

print("Attempting torch.export with StaticCache...")
try:
    from torch.export import export, Dim
    seq_dim_d = Dim("seq_len", min=1, max=MAX_CONTEXT_SIZE)
    
    exported_d = export(
        static_wrapper,
        example,
        dynamic_shapes={"input_ids": {1: seq_dim_d}},
    )
    exported_d = exported_d.run_decompositions({})
    print("✅ torch.export with StaticCache succeeded")
    
except Exception as e:
    print(f"❌ torch.export failed: {e}")
    print("StaticCache may use dynamic control flow incompatible with export.")
    print("Trying torch.jit.trace as fallback...")
    
    try:
        trace_input = torch.zeros((1, 2), dtype=torch.int32)
        traced_d = torch.jit.trace(static_wrapper, [trace_input])
        print("✅ torch.jit.trace with StaticCache succeeded")
    except Exception as e2:
        print(f"❌ torch.jit.trace also failed: {e2}")
        traced_d = None
        exported_d = None

### 6.4 Convert to Core ML with per-layer states

In [ ]:
"""
If export or trace succeeded, convert to Core ML.
Per-layer state naming: key_cache_0, value_cache_0, ..., key_cache_27, value_cache_27
Total: 56 state tensors.
"""

per_layer_shape = (1, NUM_KV_HEADS, MAX_CONTEXT_SIZE, HEAD_DIM)

states_d = []
for layer_idx in range(NUM_LAYERS):
    states_d.append(
        ct.StateType(
            wrapped_type=ct.TensorType(
                shape=per_layer_shape,
                dtype=np.float16,
            ),
            name=f"key_cache_{layer_idx}",
        )
    )
    states_d.append(
        ct.StateType(
            wrapped_type=ct.TensorType(
                shape=per_layer_shape,
                dtype=np.float16,
            ),
            name=f"value_cache_{layer_idx}",
        )
    )

print(f"Total state tensors: {len(states_d)} (expected {NUM_LAYERS * 2})")

# Convert whichever succeeded
model_to_convert = None
if 'exported_d' in dir() and exported_d is not None:
    model_to_convert = exported_d
    print("Converting from torch.export...")
elif 'traced_d' in dir() and traced_d is not None:
    model_to_convert = traced_d
    print("Converting from torch.jit.trace...")

if model_to_convert is not None:
    seq_dim_d = ct.RangeDim(lower_bound=1, upper_bound=MAX_CONTEXT_SIZE, default=1)
    
    try:
        mlmodel_d = ct.convert(
            model_to_convert,
            inputs=[ct.TensorType(shape=(1, seq_dim_d), dtype=np.int32, name="inputIds")],
            outputs=[ct.TensorType(dtype=np.float16, name="logits")],
            states=states_d,
            minimum_deployment_target=ct.target.iOS18,
            skip_model_load=True,
        )
        mlmodel_d.save("models/Qwen3_1_7B_staticcache_fp16.mlpackage")
        print("✅ Core ML conversion complete (Approach D)")
    except Exception as e:
        print(f"❌ Core ML conversion failed: {e}")
else:
    print("⚠️ No model available for Core ML conversion (Approach D)")

del static_wrapper

---

## 8. Verification & Benchmarking

### 8.1 Numerical verification (stateful approach)

In [ ]:
"""
Compare stateful Core ML model output vs PyTorch reference.
IMPORTANT: State accumulates across calls — must reset between tests.
"""

def verify_stateful_model(mlmodel_path: str, tokenizer, ref_logits: np.ndarray):
    mlmodel = ct.models.MLModel(mlmodel_path, compute_units=ct.ComputeUnit.CPU_ONLY)
    
    # Create fresh state
    state = mlmodel.make_state()
    
    # Run prefill with test tokens
    test_prompt = "Hello, how are you"
    tokens = tokenizer(test_prompt, return_tensors="np")["input_ids"].astype(np.int32)
    seq_len = tokens.shape[1]
    
    mask = build_causal_mask(seq_len, seq_len)
    
    result = mlmodel.predict(
        {"inputIds": tokens, "causalMask": mask},
        state,
    )
    
    coreml_logits = result["logits"][0, -1, :].astype(np.float32)
    
    # Compare top-K predictions
    ref_top5 = np.argsort(ref_logits)[-5:][::-1]
    cml_top5 = np.argsort(coreml_logits)[-5:][::-1]
    
    print(f"PyTorch top-5: {ref_top5}")
    print(f"CoreML  top-5: {cml_top5}")
    print(f"Top-1 match: {ref_top5[0] == cml_top5[0]}")
    
    max_diff = np.max(np.abs(ref_logits - coreml_logits))
    print(f"Max logit difference: {max_diff:.4f}")
    
    return max_diff < 1.0  # FP16 + int4 can have notable differences

### 8.2 Stateful decode loop verification

In [ ]:
def verify_stateful_decode(mlmodel_path: str, tokenizer, num_steps: int = 10):
    """
    Verify the stateful decode loop produces coherent text.
    """
    mlmodel = ct.models.MLModel(mlmodel_path, compute_units=ct.ComputeUnit.CPU_ONLY)
    state = mlmodel.make_state()
    
    prompt = "The capital of France is"
    tokens = tokenizer(prompt, return_tensors="np")["input_ids"].astype(np.int32)
    seq_len = tokens.shape[1]
    
    generated = list(tokens[0])
    
    # Prefill
    mask = build_causal_mask(seq_len, seq_len)
    result = mlmodel.predict({"inputIds": tokens, "causalMask": mask}, state)
    next_token = int(np.argmax(result["logits"][0, -1, :]))
    generated.append(next_token)
    
    # Decode
    for step in range(num_steps - 1):
        total_len = seq_len + step + 1
        if total_len >= MAX_CONTEXT_SIZE:
            print(f"Reached max context at step {step}")
            break
        
        token_input = np.array([[next_token]], dtype=np.int32)
        mask = build_causal_mask(1, total_len + 1)
        
        result = mlmodel.predict({"inputIds": token_input, "causalMask": mask}, state)
        next_token = int(np.argmax(result["logits"][0, -1, :]))
        generated.append(next_token)
        
        if next_token == tokenizer.eos_token_id:
            break
    
    text = tokenizer.decode(generated)
    print(f"Generated: {text}")
    return text

### 8.3 State reset verification

In [ ]:
def verify_state_reset(mlmodel_path: str, tokenizer):
    """
    Verify that make_state() produces independent state objects,
    and that results are deterministic when starting from fresh state.
    """
    mlmodel = ct.models.MLModel(mlmodel_path, compute_units=ct.ComputeUnit.CPU_ONLY)
    
    tokens = np.array([[1, 2, 3]], dtype=np.int32)
    mask = build_causal_mask(3, 3)
    
    # Run with state 1
    state1 = mlmodel.make_state()
    result1 = mlmodel.predict({"inputIds": tokens, "causalMask": mask}, state1)
    logits1 = result1["logits"][0, -1, :]
    
    # Run with state 2 (independent)
    state2 = mlmodel.make_state()
    result2 = mlmodel.predict({"inputIds": tokens, "causalMask": mask}, state2)
    logits2 = result2["logits"][0, -1, :]
    
    diff = np.max(np.abs(logits1.astype(np.float32) - logits2.astype(np.float32)))
    print(f"State independence check — max diff: {diff}")
    assert diff < 1e-6, "States are not independent!"
    
    # Run state 1 again — should give DIFFERENT results (cache is populated)
    result1b = mlmodel.predict(
        {"inputIds": np.array([[4]], dtype=np.int32), "causalMask": build_causal_mask(1, 4)},
        state1,
    )
    logits1b = result1b["logits"][0, -1, :]
    
    # state2 with same token should give different results (different history)
    result2b = mlmodel.predict(
        {"inputIds": np.array([[4]], dtype=np.int32), "causalMask": build_causal_mask(1, 4)},
        state2,
    )
    logits2b = result2b["logits"][0, -1, :]
    
    # They should be the same since both states saw [1,2,3] then [4]
    diff2 = np.max(np.abs(logits1b.astype(np.float32) - logits2b.astype(np.float32)))
    print(f"Same-history check — max diff: {diff2}")
    assert diff2 < 1e-6, "Same history should produce same output!"
    
    print("✅ State reset verification passed")

### 8.4 Performance benchmark

In [ ]:
import time

def benchmark_model(mlmodel_path: str, num_tokens: int = 100, use_state: bool = True):
    """
    Benchmark token generation throughput.
    """
    mlmodel = ct.models.MLModel(mlmodel_path, compute_units=ct.ComputeUnit.CPU_AND_GPU)
    
    state = mlmodel.make_state() if use_state else None
    
    # Prefill
    tokens = np.random.randint(0, VOCAB_SIZE, (1, 5), dtype=np.int32)
    mask = build_causal_mask(5, 5)
    
    predict_kwargs = {"inputIds": tokens, "causalMask": mask}
    
    if use_state:
        mlmodel.predict(predict_kwargs, state)
    else:
        predict_kwargs["keyCacheIn"] = np.zeros(kv_shape, dtype=np.float16)
        predict_kwargs["valueCacheIn"] = np.zeros(kv_shape, dtype=np.float16)
        result = mlmodel.predict(predict_kwargs)
    
    # Decode benchmark
    past_len = 5
    start = time.perf_counter()
    
    for step in range(num_tokens):
        token = np.array([[step % VOCAB_SIZE]], dtype=np.int32)
        mask = build_causal_mask(1, past_len + 1)
        
        predict_kwargs = {"inputIds": token, "causalMask": mask}
        
        if use_state:
            mlmodel.predict(predict_kwargs, state)
        else:
            predict_kwargs["keyCacheIn"] = result.get("keyCacheOut", np.zeros(kv_shape, dtype=np.float16))
            predict_kwargs["valueCacheIn"] = result.get("valueCacheOut", np.zeros(kv_shape, dtype=np.float16))
            result = mlmodel.predict(predict_kwargs)
        
        past_len += 1
    
    elapsed = time.perf_counter() - start
    tok_per_sec = num_tokens / elapsed
    ms_per_tok = (elapsed / num_tokens) * 1000
    
    print(f"Generated {num_tokens} tokens in {elapsed:.2f}s")
    print(f"  {tok_per_sec:.1f} tokens/sec")
    print(f"  {ms_per_tok:.1f} ms/token")
    
    return tok_per_sec